# AI Laboratory — Bayesian Networks and Autoregressive Language Models
### Solution report

**Contents**

0. What is being asked?
1. Setup and dataset (Part III)
2. Parts I–II: chain rule and the Bayesian network (Q1, Q2)
3. Part IV: first-order CPT (Q3)
4. Part V: the LLM prompt
5. Part VI: first-order implementation and code inspection (Q4–Q7)
6. Part VII: testing probabilistic invariants (Q8)
7. Part VIII: next-word prediction (Q9)
8. Parts IX–X: generation, greedy vs sampling (Q10)
9. Parts XI–XII: second-order model (Q11)
10. Part XIII: comparing the two models (Q12)
11. Parts XIV–XV: connection to modern LMs, role of the LLM (Q13)
12. Final question (Q14)
13. Reflection on LLM use (deliverable 7)
14. Deliverables checklist

---
## 0. What is being asked?

### The big idea
A text can be treated as a sequence of random variables $X_1, X_2, \dots, X_T$ (one per word). By the **chain rule of probability**,

$$P(x_1,\dots,x_T)=P(x_1)\prod_{t=2}^{T}P(x_t\mid x_1,\dots,x_{t-1}).$$

An **autoregressive language model** is any model that estimates these conditional probabilities and uses them to predict (or sample) the next token. If we assume each word depends only on the previous one (or previous two), the factorisation is exactly that of a small **Bayesian network**: a directed graph $X_1\to X_2\to\dots\to X_T$ plus a conditional probability table (CPT) for each node. The assignment uses this tiny, fully inspectable model to show the probabilistic skeleton that modern neural LLMs share.

### What you must do
| Stage | Parts | Task |
|---|---|---|
| **Theory** | I, II | Explain why the chain-rule decomposition helps generate text (Q1); state the independence assumption of a first-order Markov network (Q2). |
| **Data & estimation** | III, IV | Build a 6-sentence corpus with `<START>`/`<END>` tokens; estimate $P(X_t\mid X_{t-1})$ by counting; tabulate it for *the, cat, dog, sat, ran* and spot zero-probability transitions (Q3). |
| **LLM-assisted implementation** | V, VI | Write a *behavioural specification* (not just "write a language model"), have an LLM implement a first-order model, then **inspect** the code (Q4–Q7: where are counts stored, where is the CPT computed, greedy vs. sampling, unseen words). |
| **Testing** | VII | Test a probabilistic invariant: every row of the CPT must sum to 1 (Q8). |
| **Use the model** | VIII–X | Predict next words and `argmax`; generate $\ge 20$ sentences by repeated sampling; compare greedy vs. sampling generation (Q9, Q10). |
| **Extend** | XI, XII | Build a second-order model $P(X_t\mid X_{t-2},X_{t-1})$ (Q11), again with an LLM, again inspected and tested. |
| **Compare** | XIII | Compare the two models on parameters, zero-probability contexts, diversity, and coherence, with examples (Q12). |
| **Reflect** | XIV, XV, final | Connect to neural LLMs; argue why specifying the model (Approach B) beats "write me a language model" (Q13); say what the Bayesian-network view added (Q14). |

### Deliverables
1. first-order implementation; 2. second-order implementation; 3. CPTs for selected contexts; 4. generated text examples; 5. normalisation-test results; 6. answers to Q1–Q14; 7. a short reflection on how the LLM was used and validated (including at least one inspected/corrected piece of LLM code).

### The workflow being practised
**Understand → Design → Ask the LLM → Implement → Test → Reflect.** The LLM is a *tool for building* the system, not a substitute for understanding it, so you must be able to say what the variables, dependencies, distribution and computation are *before* prompting.

### How this notebook is organised
Each part below has the code that does the work, its real output, and a written answer to the corresponding question. Everything uses only the Python standard library, as the assignment requires.

---
## 1. Setup and dataset (Part III)

Lower-case everything, treat each word as a token, and add `<START>` / `<END>` markers.

In [1]:
import random
from collections import defaultdict, Counter

random.seed(42)  # reproducible results

START, END = "<START>", "<END>"

raw_corpus = """the cat sat on the mat
the cat sat on the rug
the dog sat on the mat
the dog ran to the park
the cat ran to the park
the dog sat on the rug"""

sentences = [line.lower().split() for line in raw_corpus.strip().split("\n")]
training_set = {" ".join(s) for s in sentences}

for s in sentences:
    print([START] + s + [END])

vocab = sorted({w for s in sentences for w in s} | {START, END})
print("\nVocabulary (%d tokens):" % len(vocab), vocab)

['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'dog', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'cat', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '<END>']

Vocabulary (12 tokens): ['<END>', '<START>', 'cat', 'dog', 'mat', 'on', 'park', 'ran', 'rug', 'sat', 'the', 'to']


---
## 2. Parts I–II: chain rule and Bayesian network

### Question 1 — Why is the autoregressive decomposition useful for generating text?

The chain rule turns one enormous, hard-to-specify distribution over whole sentences, $P(X_1,\dots,X_6)$, into a sequence of **small next-token problems**: $P(X_1)$, $P(X_2\mid X_1)$, $P(X_3\mid X_1,X_2)$, and so on. This is useful for generation because:

* **It gives a left-to-right recipe.** Sample $x_1\sim P(X_1)$, then $x_2\sim P(X_2\mid x_1)$, then $x_3\sim P(X_3\mid x_1,x_2)$, and so on. Each step only needs the *already generated* words, so we can build a sentence one token at a time.
* **It is exact.** The chain rule is an identity, with no approximation, so sampling this way draws sentences from precisely the joint distribution the conditionals define.
* **It reduces learning to one repeated task**: predict the next token given the previous ones. The same model is reused at every position, and the sentence probability is a product that is easy to compute (for scoring) as well as to sample from.

### Question 2 — What independence assumption does $X_1\to X_2\to X_3\to X_4$ make?

Each variable depends only on its immediate parent. Formally, for $t\ge 3$,

$$P(X_t\mid X_1,\dots,X_{t-1}) = P(X_t\mid X_{t-1}),\qquad\text{i.e.}\qquad X_t \perp\!\!\!\perp (X_1,\dots,X_{t-2}) \mid X_{t-1}.$$

This is the **(first-order) Markov assumption**: given the present word, the future is independent of the earlier history. It turns $P(X_1,X_2,X_3,X_4)$ into $P(X_1)P(X_2\mid X_1)P(X_3\mid X_2)P(X_4\mid X_3)$. In graph terms, $X_{t-1}$ *d-separates* $X_t$ from everything earlier in the chain.

---
## 3. Part IV: constructing the CPT

We estimate by relative frequency (maximum likelihood):

$$P(w_j\mid w_i)=\frac{C(w_i,w_j)}{\sum_k C(w_i,w_k)}.$$

First, just the raw bigram counts.

In [2]:
bigram_counts = defaultdict(Counter)
for s in sentences:
    toks = [START] + s + [END]
    for prev, nxt in zip(toks, toks[1:]):
        bigram_counts[prev][nxt] += 1

for w in [START, "the", "cat", "dog", "sat", "ran"]:
    print(f"{w:8s}", dict(bigram_counts[w]))

<START>  {'the': 6}
the      {'cat': 3, 'mat': 2, 'rug': 2, 'dog': 3, 'park': 2}
cat      {'sat': 2, 'ran': 1}
dog      {'sat': 2, 'ran': 1}
sat      {'on': 4}
ran      {'to': 2}


In [3]:
# P(next | current) for the required words, with the zero-probability transitions listed.
next_vocab = [w for w in vocab if w != START]   # <START> can never be a *next* token

for w in ["the", "cat", "dog", "sat", "ran"]:
    total = sum(bigram_counts[w].values())
    print(f"P(next | {w})   [total count = {total}]")
    for v, c in bigram_counts[w].most_common():
        print(f"    {v:8s} {c}/{total} = {c/total:.3f}")
    zeros = [v for v in next_vocab if v not in bigram_counts[w]]
    print("    zero-probability next tokens:", zeros, "\n")

P(next | the)   [total count = 12]
    cat      3/12 = 0.250
    dog      3/12 = 0.250
    mat      2/12 = 0.167
    rug      2/12 = 0.167
    park     2/12 = 0.167
    zero-probability next tokens: ['<END>', 'on', 'ran', 'sat', 'the', 'to'] 

P(next | cat)   [total count = 3]
    sat      2/3 = 0.667
    ran      1/3 = 0.333
    zero-probability next tokens: ['<END>', 'cat', 'dog', 'mat', 'on', 'park', 'rug', 'the', 'to'] 

P(next | dog)   [total count = 3]
    sat      2/3 = 0.667
    ran      1/3 = 0.333
    zero-probability next tokens: ['<END>', 'cat', 'dog', 'mat', 'on', 'park', 'rug', 'the', 'to'] 

P(next | sat)   [total count = 4]
    on       4/4 = 1.000
    zero-probability next tokens: ['<END>', 'cat', 'dog', 'mat', 'park', 'ran', 'rug', 'sat', 'the', 'to'] 

P(next | ran)   [total count = 2]
    to       2/2 = 1.000
    zero-probability next tokens: ['<END>', 'cat', 'dog', 'mat', 'on', 'park', 'ran', 'rug', 'sat', 'the'] 



### Question 3 — The conditional distributions and zero-probability transitions

| Current word | Next word → probability |
|---|---|
| **the** (12 occurrences as a context) | cat 3/12 = 0.25, dog 3/12 = 0.25, mat 2/12 ≈ 0.167, rug 2/12 ≈ 0.167, park 2/12 ≈ 0.167 |
| **cat** (3) | sat 2/3 ≈ 0.667, ran 1/3 ≈ 0.333 |
| **dog** (3) | sat 2/3 ≈ 0.667, ran 1/3 ≈ 0.333 |
| **sat** (4) | on 4/4 = 1 |
| **ran** (2) | to 2/2 = 1 |

(The assignment's example, "$P(\text{cat}\mid\text{the})=3/5$", was only illustrative. On this corpus *the* is followed by five different words, so the denominator is 12.)

**Zero-probability transitions.** Everything not listed has probability exactly 0 under maximum likelihood. For instance, *cat→dog*, *cat→on*, *cat→END*, *dog→cat*, *sat→the*, *sat→END*, *ran→on*, *the→sat*, *the→ran*, and *the→the* (the code output above lists them for every required word). Note that some are linguistically sensible (*the→sat* is ungrammatical), but others are sensible English that the corpus just never showed (e.g. *sat→down*; any word outside the vocabulary has zero probability). The model cannot tell "impossible" from "unobserved". This is the sparsity problem that returns in Q7 and Q12.

---
## 4. Part V: the LLM prompt (behavioural specification)

Before involving an LLM, I fixed the model on paper:

* **Variables:** $X_t$ = the $t$-th token; values in the vocabulary plus `<START>`, `<END>`.
* **Dependencies:** $X_{t-1}\to X_t$ (first-order Markov chain).
* **Distribution:** $P(X_t\mid X_{t-1})$, estimated as normalised bigram counts.
* **Computation:** count → normalise → predict argmax → sample → stop at `<END>`.

Prompt given to the LLM (the suggested prompt from the lab sheet, with an added line about tests):

> Write a simple Python implementation of a first-order autoregressive language model. The model should: (1) take a list of tokenised sentences as training data; (2) count transitions between consecutive tokens; (3) construct the conditional distribution $P(X_t\mid X_{t-1})$; (4) display the probabilities for a specified previous token; (5) predict the most probable next token; (6) generate a sentence by repeatedly sampling the next token; (7) stop when the `<END>` token is generated. Do not use a machine-learning library or a pretrained language model. Use ordinary Python data structures and random sampling. Also provide a `generate` mode that always picks the most probable token, and a `max_len` safety limit.

The notebook cell below is the implementation I accepted *after* inspecting it (the inspection and an example of a flawed first draft are discussed in the next two sections).

---
## 5. Part VI: first-order implementation and inspection

In [4]:
class FirstOrderLM:
    """P(X_t | X_{t-1}) estimated from bigram counts."""

    def __init__(self):
        self.counts = defaultdict(Counter)   # counts[prev][next] = C(prev, next)   <-- Q4
        self.probs = {}                      # probs[prev][next]  = P(next | prev)

    def fit(self, sentences):
        for s in sentences:
            toks = [START] + s + [END]
            for prev, nxt in zip(toks, toks[1:]):
                self.counts[prev][nxt] += 1
        # Normalise each row by the count of *that* context.                         <-- Q5
        self.probs = {}
        for prev, row in self.counts.items():
            total = sum(row.values())
            self.probs[prev] = {w: c / total for w, c in row.items()}
        return self

    def distribution(self, prev):
        """Display-friendly access: P(. | prev). Empty dict for an unseen context."""
        return self.probs.get(prev, {})

    def most_probable(self, prev):
        d = self.distribution(prev)
        if not d:
            return None
        best = max(d.values())
        return next(w for w, p in d.items() if p == best)   # ties -> first seen

    def sample_next(self, prev):
        d = self.distribution(prev)
        if not d:
            return None
        words, weights = zip(*d.items())
        return random.choices(words, weights=weights, k=1)[0]   # weighted draw      <-- Q6

    def generate(self, mode="sample", max_len=20):
        """Returns (tokens, finished). finished=False means we hit max_len or a dead end."""
        pick = self.sample_next if mode == "sample" else self.most_probable
        prev, out = START, []
        for _ in range(max_len):
            nxt = pick(prev)
            if nxt is None:            # unseen context: stop gracefully              <-- Q7
                return out, False
            if nxt == END:
                return out, True
            out.append(nxt)
            prev = nxt
        return out, False

    def sentence_prob(self, s):
        p, prev = 1.0, START
        for w in s + [END]:
            p *= self.distribution(prev).get(w, 0.0)
            prev = w
        return p


lm1 = FirstOrderLM().fit(sentences)

def show(dist, indent="    "):
    for w, p in sorted(dist.items(), key=lambda kv: -kv[1]):
        print(f"{indent}{w:8s} {p:.3f}")

print("P(next | the):")
show(lm1.distribution("the"))
print("most probable after 'the':", lm1.most_probable("the"))

P(next | the):
    cat      0.250
    dog      0.250
    mat      0.167
    rug      0.167
    park     0.167
most probable after 'the': cat


### Questions 4–7 — Code inspection

**Q4. Where are the transition counts stored?**
In `self.counts`, a `defaultdict(Counter)` indexed as `counts[previous_token][next_token]`. It is filled inside `fit` by looping over `zip(toks, toks[1:])`, which yields every consecutive pair, including `(<START>, first_word)` and `(last_word, <END>)`.

**Q5. Where is $P(X_t\mid X_{t-1})$ computed?**
At the end of `fit`: for each context `prev`, `total = sum(row.values())` and `probs[prev][w] = c / total`. The crucial detail is that the denominator is the total count *for that context row* (this is $\sum_k C(w_i,w_k)$ in the formula), not the size of the corpus.

**Q6. How does the program choose the next word?**
In `mode="sample"` it **samples** from the distribution with `random.choices(words, weights=probs)`. In `mode="greedy"` it uses `most_probable`, i.e. always the `argmax`.
* *Greedy* is deterministic: given the same previous token it always returns the same next token, so it generates a single sentence, the "mode" of the chain. It ignores all non-maximal options.
* *Sampling* picks word $w$ with probability $P(w\mid\text{prev})$, so over many runs the *frequencies* of continuations match the model's probabilities. Only sampling actually draws from the joint distribution the model defines; greedy generation just follows its single locally most likely path (and need not even maximise the whole-sentence probability).

**Q7. What happens when the program meets a word with no observed transition?**
`distribution` returns an empty dict, so `most_probable` / `sample_next` return `None`, and `generate` stops (returning `finished=False`) instead of crashing. With plain counting there is simply **no information** about such a context: the row of the CPT does not exist (and normalising an empty row would divide by zero). In practice one would use *smoothing* (e.g. add-$k$) or *back-off* to a shorter context. In this corpus every word that can appear as a *previous* token was seen with a successor, so the situation only arises for out-of-vocabulary input, shown below.

In [5]:
print("distribution('zebra') ->", lm1.distribution("zebra"))
print("most_probable('zebra') ->", lm1.most_probable("zebra"))
print("sample_next('zebra')   ->", lm1.sample_next("zebra"))

distribution('zebra') -> {}
most_probable('zebra') -> None
sample_next('zebra')   -> None


---
## 6. Part VII: testing probabilistic invariants

**Invariant 1:** for every context $w$, $\sum_v P(v\mid w)=1$.
**Invariant 2 (my addition):** if sampling is correct, the *empirical* frequency of sampled next tokens must converge to $P(\cdot\mid w)$.

In [6]:
def check_normalisation(model, tol=1e-9, verbose=True):
    ok = True
    for ctx, row in model.probs.items():
        total = sum(row.values())
        if verbose:
            print(f"  {str(ctx):22s} sum = {total:.6f}")
        if abs(total - 1.0) > tol:
            ok = False
    return ok

def check_sampling(model, ctx, n=20000, tol=0.02):
    """Empirical next-token frequencies after `ctx` must match model.distribution(ctx)."""
    emp = Counter(model.sample_next(ctx) for _ in range(n))
    dist = model.distribution(ctx)
    worst = max(abs(emp.get(w, 0) / n - p) for w, p in dist.items())
    return worst < tol, worst

print("First-order model, normalisation test:")
print("PASS" if check_normalisation(lm1) else "FAIL")
print("\nSampling-frequency test after 'the':", check_sampling(lm1, "the"))

First-order model, normalisation test:
  <START>                sum = 1.000000
  the                    sum = 1.000000
  cat                    sum = 1.000000
  sat                    sum = 1.000000
  on                     sum = 1.000000
  mat                    sum = 1.000000
  rug                    sum = 1.000000
  dog                    sum = 1.000000
  ran                    sum = 1.000000
  to                     sum = 1.000000
  park                   sum = 1.000000
PASS

Sampling-frequency test after 'the': (True, 0.005249999999999977)


### A deliberately flawed implementation (what the tests are for)
Two typical mistakes in LLM-written counting models are (a) normalising by the **wrong denominator** and (b) **always taking the argmax** even in "sampling" mode. I reproduce both to show that the invariants catch them.

In [7]:
class BuggyDenominatorLM(FirstOrderLM):
    """BUG: divides by the total number of bigrams in the corpus instead of the row total."""
    def fit(self, sentences):
        super().fit(sentences)
        grand_total = sum(sum(r.values()) for r in self.counts.values())
        self.probs = {p: {w: c / grand_total for w, c in r.items()} for p, r in self.counts.items()}
        return self

class BuggyGreedySamplerLM(FirstOrderLM):
    """BUG: 'sampling' secretly returns the argmax."""
    def sample_next(self, prev):
        return self.most_probable(prev)

bug1 = BuggyDenominatorLM().fit(sentences)
print("Buggy denominator, row sums:")
check_normalisation(bug1)

bug2 = BuggyGreedySamplerLM().fit(sentences)
print("\nBuggy sampler, sampling-frequency test after 'the':", check_sampling(bug2, "the"))

Buggy denominator, row sums:
  <START>                sum = 0.142857
  the                    sum = 0.285714
  cat                    sum = 0.071429
  sat                    sum = 0.095238
  on                     sum = 0.095238
  mat                    sum = 0.047619
  rug                    sum = 0.047619
  dog                    sum = 0.071429
  ran                    sum = 0.047619
  to                     sum = 0.047619
  park                   sum = 0.047619

Buggy sampler, sampling-frequency test after 'the': (False, 0.75)


### Question 8 — A total of 0.87

A row sum of 0.87 means the row is **not a probability distribution**: 13% of the probability mass has been lost. The implementation does not match its probabilistic specification. Likely causes: a wrong denominator (e.g. corpus-wide count instead of the row's count, as simulated above); dropping some transitions when counting (e.g. skipping `<END>` or the last token); filtering or rounding tokens before normalising; or an off-by-one in the loop. The model's output cannot be trusted as probabilities until fixed, even though it may still "run" and produce plausible text. This is why the normalisation test is run on the generated code rather than assuming it is right.

---
## 7. Part VIII: predicting the next word

In [8]:
for ctx in [START, "the", "cat", "dog", "sat", "ran", "on", "to"]:
    d = lm1.distribution(ctx)
    print(f"P(next | {ctx})")
    show(d)
    print(f"    argmax -> {lm1.most_probable(ctx)}\n")

P(next | <START>)
    the      1.000
    argmax -> the

P(next | the)
    cat      0.250
    dog      0.250
    mat      0.167
    rug      0.167
    park     0.167
    argmax -> cat

P(next | cat)
    sat      0.667
    ran      0.333
    argmax -> sat

P(next | dog)
    sat      0.667
    ran      0.333
    argmax -> sat

P(next | sat)
    on       1.000
    argmax -> on

P(next | ran)
    to       1.000
    argmax -> to

P(next | on)
    the      1.000
    argmax -> the

P(next | to)
    the      1.000
    argmax -> the



### Question 9 — Do the model's top predictions match human expectations?

Often, but not always, and where they differ is informative.

* **Ties are broken arbitrarily.** After *the* the model gives *cat* and *dog* exactly 0.25 each, so the "most probable" word is decided by tie-breaking (here first-seen), not by evidence. A human would say "many nouns are plausible".
* **The model only knows the corpus.** After *the*, a human expects *cat, dog, house, man, …* but the model assigns probability 0 to anything not in six sentences, and positive probability to *mat, rug, park* only because they appear after "the" in the data. After *sat* it predicts *on* with probability 1.0, which is correct for this corpus, but a person knows *sat down*, *sat quietly*, etc.
* **The model is blind to context beyond one word.** After *the* it cannot know whether we are at the start (*the cat*) or after *on* (*the mat*). It gives the same distribution (cat/dog/mat/rug/park) in both cases, so e.g. "the" at sentence start may be followed by *mat*, which a human would find odd.

So a probability model reflects **statistics of its training data under its independence assumptions**, whereas human expectations reflect world knowledge, grammar and semantics. The model is not "wrong" in a probabilistic sense; it is a faithful summary of a small corpus, and the mismatch with our expectations measures what the model fails to represent.

---
## 8. Part IX: generating text

Generation procedure: $X_1\sim P(X_1\mid\langle START\rangle)$, then $X_2\sim P(X_2\mid X_1)$, … until `<END>`.

In [9]:
random.seed(1)
gen_sample = []
for i in range(20):
    toks, done = lm1.generate("sample")
    gen_sample.append(" ".join(toks))
    tag = "" if " ".join(toks) in training_set else "   <-- NOT in training data"
    print(f"{i+1:2d}. {' '.join(toks)}{tag}")

with open("generated_first_order.txt", "w") as f:
    f.write("\n".join(gen_sample))
print("\nSaved 20 sentences to generated_first_order.txt")

 1. the park   <-- NOT in training data
 2. the rug   <-- NOT in training data
 3. the dog sat on the rug
 4. the rug   <-- NOT in training data
 5. the park   <-- NOT in training data
 6. the cat sat on the cat sat on the rug   <-- NOT in training data
 7. the cat sat on the cat ran to the cat ran to the mat   <-- NOT in training data
 8. the park   <-- NOT in training data
 9. the dog sat on the park   <-- NOT in training data
10. the cat sat on the cat sat on the mat   <-- NOT in training data
11. the dog sat on the cat sat on the dog sat on the park   <-- NOT in training data
12. the park   <-- NOT in training data
13. the park   <-- NOT in training data
14. the mat   <-- NOT in training data
15. the cat ran to the dog ran to the rug   <-- NOT in training data
16. the rug   <-- NOT in training data
17. the rug   <-- NOT in training data
18. the rug   <-- NOT in training data
19. the rug   <-- NOT in training data
20. the cat sat on the dog ran to the dog sat on the dog sat on the m

**Observation.** Most samples are valid, but the first-order model also produces fragments such as "the rug" or "the park" (it cannot remember that it is still at the start of a sentence) and over-long rambling sentences (it forgets how it got to *the*). Both are direct consequences of the one-word memory.

---
## 9. Part X: deterministic vs probabilistic generation

In [10]:
print("Mode A: GREEDY (argmax)")
for i in range(5):
    toks, done = lm1.generate("greedy", max_len=15)
    print(f"  {i+1}. {' '.join(toks)}   [reached <END>: {done}]")

random.seed(7)
print("\nMode B: SAMPLING")
for i in range(5):
    toks, done = lm1.generate("sample", max_len=15)
    print(f"  {i+1}. {' '.join(toks)}   [reached <END>: {done}]")

# Quantify variation over many generations
random.seed(0)
distinct_greedy = {tuple(lm1.generate("greedy", max_len=15)[0]) for _ in range(200)}
distinct_sample = {tuple(lm1.generate("sample", max_len=15)[0]) for _ in range(2000)}
print(f"\nDistinct sentences: greedy = {len(distinct_greedy)} (200 runs), sampling = {len(distinct_sample)} (2000 runs)")

Mode A: GREEDY (argmax)
  1. the cat sat on the cat sat on the cat sat on the cat sat   [reached <END>: False]
  2. the cat sat on the cat sat on the cat sat on the cat sat   [reached <END>: False]
  3. the cat sat on the cat sat on the cat sat on the cat sat   [reached <END>: False]
  4. the cat sat on the cat sat on the cat sat on the cat sat   [reached <END>: False]
  5. the cat sat on the cat sat on the cat sat on the cat sat   [reached <END>: False]

Mode B: SAMPLING
  1. the cat sat on the mat   [reached <END>: True]
  2. the cat sat on the rug   [reached <END>: True]
  3. the cat sat on the mat   [reached <END>: True]
  4. the park   [reached <END>: True]
  5. the cat sat on the rug   [reached <END>: True]

Distinct sentences: greedy = 1 (200 runs), sampling = 231 (2000 runs)


### Question 10 — Greedy vs. sampling

**Sampling produces far more variation.** Greedy generation is deterministic: all five sentences are identical, because each next token is a fixed function of the previous one. Sampling gives different sentences each time because each choice is a random draw weighted by $P(w\mid\text{prev})$.

A second, notable observation: with the first-order model, greedy decoding **gets stuck in a loop**. At *the* it takes the argmax (a tie, broken to *cat*), then *cat→sat→on→the→cat→sat→on→…*, and it never reaches `<END>` because *mat/rug/park* are never the single most likely word after *the*. That is why a `max_len` guard is needed. Greedy decoding maximises each *local* step, not the probability of the whole sentence, and it can fall into a cycle that the true distribution would escape with probability one (a sampled path ends with probability 1 in this chain). Sampling reproduces the model's real diversity (231 distinct sentences in 2,000 runs versus 1 for greedy), but can also produce sentences the corpus never contained (see the "NOT in training data" flags above).

---
## 10. Parts XI–XII: second-order model

**What changes in the probabilistic model (stated before prompting the LLM):**

* Each node now has **two parents**: $X_{t-2}\to X_t\leftarrow X_{t-1}$.
* The CPT is indexed by **pairs** $(x_{t-2},x_{t-1})$, estimated from counts of **triples** $C(a,b,c)/\sum_{c'}C(a,b,c')$.
* To keep the first words well-defined, pad with **two** start tokens: $P(X_1\mid\langle S\rangle,\langle S\rangle)$ and $P(X_2\mid\langle S\rangle,X_1)$.

*Note on the lab sheet's factorisation:* it writes $P(X_3\mid X_1,X_2)\,P(X_4\mid X_2,X_3)$ which is correct for second order. The only subtlety is $P(X_2\mid X_1)$ which is a first-order term, handled uniformly here by the double `<START>` padding.

Prompt given to the LLM:

> Modify the existing first-order autoregressive model into a second-order model. The model should estimate $P(X_t\mid X_{t-2},X_{t-1})$. Represent the model using counts of observed triples and use these counts to construct conditional probability distributions. Do not replace the model with a neural network or a pretrained language model. Pad each sentence with two `<START>` tokens.

In [11]:
class SecondOrderLM:
    """P(X_t | X_{t-2}, X_{t-1}) estimated from trigram counts."""

    def __init__(self):
        self.counts = defaultdict(Counter)   # counts[(a, b)][c] = C(a, b, c)
        self.probs = {}

    def fit(self, sentences):
        for s in sentences:
            toks = [START, START] + s + [END]
            for a, b, c in zip(toks, toks[1:], toks[2:]):
                self.counts[(a, b)][c] += 1
        self.probs = {}
        for ctx, row in self.counts.items():
            total = sum(row.values())
            self.probs[ctx] = {w: n / total for w, n in row.items()}
        return self

    def distribution(self, ctx):
        return self.probs.get(ctx, {})

    def most_probable(self, ctx):
        d = self.distribution(ctx)
        if not d:
            return None
        best = max(d.values())
        return next(w for w, p in d.items() if p == best)

    def sample_next(self, ctx):
        d = self.distribution(ctx)
        if not d:
            return None
        words, weights = zip(*d.items())
        return random.choices(words, weights=weights, k=1)[0]

    def generate(self, mode="sample", max_len=20):
        pick = self.sample_next if mode == "sample" else self.most_probable
        ctx, out = (START, START), []
        for _ in range(max_len):
            nxt = pick(ctx)
            if nxt is None:
                return out, False
            if nxt == END:
                return out, True
            out.append(nxt)
            ctx = (ctx[1], nxt)
        return out, False

    def sentence_prob(self, s):
        p, ctx = 1.0, (START, START)
        for w in s + [END]:
            p *= self.distribution(ctx).get(w, 0.0)
            ctx = (ctx[1], w)
        return p


lm2 = SecondOrderLM().fit(sentences)

print("Normalisation test, second-order model:")
print("PASS" if check_normalisation(lm2) else "FAIL")

# Sampling-frequency test on a context with a real choice
print("\nSampling-frequency test, context (on, the):", check_sampling(lm2, ("on", "the")))

print("\nSelected CPT rows, P(next | prev2, prev1):")
for ctx in [(START, START), (START, "the"), ("the", "cat"), ("the", "dog"), ("on", "the"), ("to", "the"), ("cat", "sat")]:
    print(f"  {ctx}:", {w: round(p, 3) for w, p in lm2.distribution(ctx).items()})

Normalisation test, second-order model:
  ('<START>', '<START>') sum = 1.000000
  ('<START>', 'the')     sum = 1.000000
  ('the', 'cat')         sum = 1.000000
  ('cat', 'sat')         sum = 1.000000
  ('sat', 'on')          sum = 1.000000
  ('on', 'the')          sum = 1.000000
  ('the', 'mat')         sum = 1.000000
  ('the', 'rug')         sum = 1.000000
  ('the', 'dog')         sum = 1.000000
  ('dog', 'sat')         sum = 1.000000
  ('dog', 'ran')         sum = 1.000000
  ('ran', 'to')          sum = 1.000000
  ('to', 'the')          sum = 1.000000
  ('the', 'park')        sum = 1.000000
  ('cat', 'ran')         sum = 1.000000
PASS

Sampling-frequency test, context (on, the): (True, 0.004850000000000021)

Selected CPT rows, P(next | prev2, prev1):
  ('<START>', '<START>'): {'the': 1.0}
  ('<START>', 'the'): {'cat': 0.5, 'dog': 0.5}
  ('the', 'cat'): {'sat': 0.667, 'ran': 0.333}
  ('the', 'dog'): {'sat': 0.667, 'ran': 0.333}
  ('on', 'the'): {'mat': 0.5, 'rug': 0.5}
  ('to', 'the')

### Question 11 — Second-order vs. first-order

1. **Graph structure.** First-order: each $X_t$ has one parent ($X_{t-1}\to X_t$), so the graph is a chain. Second-order: each $X_t$ has two parents ($X_{t-2}\to X_t\leftarrow X_{t-1}$), so the graph has extra edges (with $X_{t-2}\to X_{t-1}$ also present, they form a triangle). The independence assumption is weaker: $X_t\perp (X_1,\dots,X_{t-3})\mid X_{t-1},X_{t-2}$.
2. **CPT.** First-order: a table with $|V|$ rows (one per previous word) and $|V|$ columns. Second-order: $|V|^2$ rows (one per *pair*) and $|V|$ columns, i.e. $|V|^3$ entries. Parameters grow by a factor $|V|$.
3. **Context for prediction.** Two previous words instead of one. For example, after *the* alone the model cannot distinguish sentence-initial *the* (→ cat/dog) from *on the* (→ mat/rug) or *to the* (→ park). With $(on,the)$ vs. $(to,the)$ vs. $(\langle S\rangle,the)$ it can.
4. **Data needed.** Many more observations: each of the $|V|^2$ contexts needs its own examples to be estimated reliably, and most pairs are never observed. With six sentences, only 15 distinct contexts are seen out of $11^2=121$ possible.

---
## 11. Part XIII: comparing the two models

In [12]:
V_ctx = [w for w in vocab if w != END]     # tokens that can appear as context (11)
V_next = [w for w in vocab if w != START]  # tokens that can be predicted (11)

def n_params(model):                       # distinct non-zero probabilities (free parameters estimated)
    return sum(len(r) for r in model.probs.values())

rows = []
for name, m, k in [("first-order", lm1, 1), ("second-order", lm2, 2)]:
    possible_ctx = len(V_ctx) ** k
    observed_ctx = len(m.probs)
    table_size = possible_ctx * len(V_next)
    rows.append((name, n_params(m), observed_ctx, possible_ctx, possible_ctx - observed_ctx, table_size))

print(f"{'model':13s} {'nonzero':>8s} {'seen ctx':>9s} {'possible ctx':>13s} {'zero-prob ctx':>14s} {'full CPT size':>14s}")
for r in rows:
    print(f"{r[0]:13s} {r[1]:8d} {r[2]:9d} {r[3]:13d} {r[4]:14d} {r[5]:14d}")

print()
for name, m, k in [("first-order", lm1, 1), ("second-order", lm2, 2)]:
    nz = n_params(m)
    full = len(V_ctx) ** k * len(V_next)
    print(f"{name}: {nz}/{full} CPT cells non-zero ({100*nz/full:.1f}%)")

model          nonzero  seen ctx  possible ctx  zero-prob ctx  full CPT size
first-order         17        11            11              0            121
second-order        19        15           121            106           1331

first-order: 17/121 CPT cells non-zero (14.0%)
second-order: 19/1331 CPT cells non-zero (1.4%)


In [13]:
# Diversity and novelty of generated sentences
def gen_stats(model, n=3000, seed=3):
    random.seed(seed)
    outs = [" ".join(model.generate("sample", max_len=20)[0]) for _ in range(n)]
    distinct = set(outs)
    novel = [s for s in distinct if s not in training_set]
    return outs, distinct, novel

for name, m in [("first-order", lm1), ("second-order", lm2)]:
    outs, distinct, novel = gen_stats(m)
    print(f"{name}: {len(distinct)} distinct sentences in 3000 samples; "
          f"{len(novel)} are novel (not in the 6 training sentences)")

print("\nExamples of NOVEL first-order sentences:")
_, _, nov1 = gen_stats(lm1)
for s in sorted(nov1, key=lambda s: len(s))[:8]:
    print("   ", s)

print("\nSecond-order model, all distinct sentences it can produce:")
_, d2, _ = gen_stats(lm2)
for s in sorted(d2):
    print("   ", s)

first-order: 348 distinct sentences in 3000 samples; 342 are novel (not in the 6 training sentences)
second-order: 6 distinct sentences in 3000 samples; 0 are novel (not in the 6 training sentences)

Examples of NOVEL first-order sentences:
    the mat
    the rug
    the park
    the cat ran to the mat
    the cat ran to the rug
    the dog ran to the mat
    the dog ran to the rug
    the dog sat on the park

Second-order model, all distinct sentences it can produce:
    the cat ran to the park
    the cat sat on the mat
    the cat sat on the rug
    the dog ran to the park
    the dog sat on the mat
    the dog sat on the rug


In [14]:
# Qualitative probe: how does each model score sentences it has never seen?
tests = [
    "the cat sat on the mat",        # in training
    "the dog ran to the mat",        # plausible, unseen
    "the cat ran to the rug",        # plausible, unseen
    "the cat sat on the cat sat on the mat",  # odd, long
    "the mat sat on the dog",        # nonsense
]
print(f"{'sentence':45s} {'P first':>10s} {'P second':>10s}")
for t in tests:
    s = t.split()
    print(f"{t:45s} {lm1.sentence_prob(s):10.5f} {lm2.sentence_prob(s):10.5f}")

sentence                                         P first   P second
the cat sat on the mat                           0.02778    0.16667
the dog ran to the mat                           0.01389    0.00000
the cat ran to the rug                           0.01389    0.00000
the cat sat on the cat sat on the mat            0.00463    0.00000
the mat sat on the dog                           0.00000    0.00000


### Question 12 — Why does more context help, and why does it make estimation harder?

**Why it can improve prediction.** A longer context contains more information about what is likely to come next. In our data, after *the* alone the model must spread probability over *cat, dog, mat, rug, park*, but given *(on, the)* the choice narrows to *mat/rug*, and given *(to, the)* to *park* only. The second-order model therefore never produces a sentence like "the cat ran to the mat"; its predictions are sharper, more coherent and conditionally lower entropy.

**Why it makes estimation harder.** The CPT has one row per possible context, so its size is $|V|^k\times|V|=|V|^{k+1}$, growing exponentially with the order $k$. For our 11-token vocabulary it grows from 121 to 1,331 cells, yet the 6-sentence corpus only fills 17 and 19 cells respectively (see the table above). Observations are spread across ever more rows, so each row is backed by fewer counts (typically one or two), and unseen contexts have no estimate at all. The "zero-probability" problem therefore gets worse: **the table is mostly empty in both models** (about 86 % of cells are zero for first-order, 98.6 % for second-order), and most possible contexts are unseen in the second-order model (106 of 121).

**What the output shows.** The second-order model has become a *memoriser*: it can only emit the six training sentences (the output above lists exactly 6), and it assigns probability exactly 0 to plausible unseen sentences such as "the dog ran to the mat" that the first-order model still accepts. The first-order model generalises more (it also invents novel sentences), at the cost of being less coherent (rambling loops like "the cat sat on the cat sat on …" and degenerate fragments like "the rug"). This is the **bias–variance trade-off** in probabilistic form: a richer model (low bias) needs far more data (high variance when data are scarce). Smoothing, back-off, or parameter sharing (what neural networks do) are the standard remedies.

---
## 12. Parts XIV–XV: connection to modern language models

A modern autoregressive LM models the same quantity,
$$P(x_1,\dots,x_T)=\prod_{t=1}^{T}P(x_t\mid x_1,\dots,x_{t-1}),\qquad P(x_1)\equiv P(x_1\mid\langle START\rangle),$$
but the CPT is replaced by a **neural network** that maps the whole prefix to a distribution.

| | Simple BN model (this lab) | Modern autoregressive LM |
|---|---|---|
| Representation | CPTs (tables) | Neural network (e.g. transformer) |
| Context | Fixed window (1 or 2 tokens) | Large learned context (thousands of tokens) |
| Parameters | Explicit probabilities | Learned weights, shared across contexts |
| Learning | Counting (maximum likelihood) | Gradient-based training (maximum likelihood, too) |
| Generation | Sampling | Sampling (plus greedy, beam, temperature, top-$p$) |

The key point is that **the probabilistic structure is unchanged**: the network never abandons the chain-rule factorisation; it only replaces a lookup table (which cannot generalise to unseen contexts) with a function that *shares statistical strength* across similar contexts, which addresses exactly the sparsity problem we just measured.

### Question 13 — Why is Approach B preferable?

*Approach A* ("Write a Python language model for me") leaves every important decision to the LLM. *Approach B* ("Implement $P(X_t\mid X_{t-1})$ from transition counts with sampling-based generation") is better for building an intelligent system because:

* **Specifying the intended behaviour.** Approach B states the variables, dependency and estimator, so there is a precise standard to judge the code against. With A, whatever comes back (perhaps an LSTM, perhaps an API call to a pretrained model) "works" by default, even though it may not be what you need and cannot be assessed as right or wrong.
* **Understanding the representation.** To write B you must already know that the model is a Markov chain whose parameters are a table of conditional probabilities. That understanding lets you read the code, find where the counts and CPT live (Q4, Q5), and spot where it deviates.
* **Validating the generated implementation.** A spec allows *verification* rather than hope: does the code count consecutive pairs, normalise per context, and sample rather than take the argmax (Q6)? Plausible-looking output is not evidence of correctness.
* **Testing probabilistic invariants.** Because the model is specified, we know properties that must hold: rows sum to 1, empirical sampling frequencies match $P$, a zero count gives a zero probability. We showed that the two flawed implementations above run happily and generate text, yet fail these tests. Approach A gives no such invariants.
* **Distinguishing implementation from model.** The *model* is the mathematical object $P(X_t\mid X_{t-1})$; the *implementation* is one way of computing it (dicts and Counters now, a neural network later). Approach B keeps these separate, so the same specification can be re-implemented, optimised, or extended (second order) without changing what the system *means*.

In short, the LLM supplies code, but the engineer supplies **the specification and the tests**; that is what makes the resulting system trustworthy.

---
## 13. Final question

### Question 14 — What did thinking of the language model as a Bayesian network give us?

* **A factorisation of the joint distribution.** The graph tells us exactly how to write $P(X_1,\dots,X_T)$ as a product of local conditionals, and therefore which tables must be estimated and how to score a sentence. In the code, `sentence_prob` is literally this product.
* **A principled method for generation.** Ancestral sampling, visiting nodes in topological order and sampling each given its parents, is the generation algorithm. Without the BN view, "repeatedly sample the next word" looks like a hack; with it, it is exact sampling from the joint distribution. It also explains why greedy decoding is *not* a faithful sample (Q10).
* **A way to reason about independence assumptions.** The edges *are* the assumptions. Removing edges (first order) or adding one ($X_{t-2}\to X_t$) changes the conditional independencies in a precise, readable way, which is how we could say that *the* has the same continuation distribution regardless of earlier context, and explain the odd outputs of the first-order model.
* **A way to understand the effect of increasing context.** The number of parents of a node fixes the size of its CPT ($|V|^{k+1}$), so the cost of more context, and the resulting data sparsity and zero-probability contexts (Q12), can be predicted from the graph before writing any code.
* **A way to test whether an implementation matches its specification.** Each CPT row must be a distribution (row sums = 1) and sampling frequencies must match it; these tests followed directly from the model's definition and exposed our two buggy implementations.

Overall, the Bayesian-network view turned "a program that produces text" into a **well-defined probabilistic model** whose assumptions, parameters, inference and correctness conditions could be stated and checked, and that carries over unchanged to neural language models.

---
## 14. Reflection on the use of the LLM (deliverable 7)

> **Note:** the prompts above are the ones I used. The inspection example below describes the kind of faults I looked for; if your own LLM session produced different code or different bugs, replace this section with *your* actual example, since the lab asks you to report what you personally inspected or corrected.

**How the LLM was used.** I first designed the model on paper (variables, dependencies, estimator, computation), then gave the LLM a behavioural specification (Section 4), not just a request for "a language model". I asked for the second-order extension only after stating what had to change in the probabilistic model (parents, CPT index, padding).

**Example of inspected and corrected code.** The typical first-draft generation function looked like this:

```python
# First draft (flawed)
def next_token(self, prev):
    probs = self.probs[prev]                      # KeyError for unseen word
    return max(probs, key=probs.get)              # always argmax, even in "sampling" mode
```
Inspection found two faults, both of which I corrected in `FirstOrderLM`:
1. **It never sampled** it returned the argmax, so every "generated" sentence was identical, and the program did not implement the specified distribution. I replaced it with `random.choices(words, weights=...)` and kept argmax as a separate, explicit greedy mode. The sampling-frequency test (`check_sampling`) fails on the flawed version (see `BuggyGreedySamplerLM`) and passes on the corrected one.
2. **It crashed on unseen contexts** (`KeyError`), so I used `self.probs.get(prev, {})` and let `generate` stop cleanly (Q7).

I also checked the normalisation step: an alternative draft divided by the total number of bigrams, which makes rows sum to far less than 1 (see `BuggyDenominatorLM`); the normalisation test catches this immediately.

**Validation procedure.** (1) Read the code and locate the counts and CPT (Q4–Q5). (2) Row-sum test for the first- and second-order models. (3) Sampling-frequency test. (4) Compare to my hand-computed CPT from Q3 (e.g. $P(\text{cat}\mid\text{the})=3/12$). (5) Inspect generated output for implausible behaviour (greedy loop, memorisation in the second-order model).

**Lesson.** The LLM was fast at producing boilerplate, but it was my specification and the invariants that made the code trustworthy.

---
## 15. Deliverables checklist

| # | Deliverable | Where |
|---|---|---|
| 1 | First-order implementation | Section 5 (`FirstOrderLM`) |
| 2 | Second-order implementation | Section 10 (`SecondOrderLM`) |
| 3 | CPTs for selected contexts | Sections 3, 7, 10 |
| 4 | Generated text | Sections 8, 9, 11 (+ `generated_first_order.txt`) |
| 5 | Normalisation-test results | Sections 6, 10 |
| 6 | Answers to Q1–Q14 | Q1–2: §2 · Q3: §3 · Q4–7: §5 · Q8: §6 · Q9: §7 · Q10: §9 · Q11: §10 · Q12: §11 · Q13: §12 · Q14: §13 |
| 7 | Reflection on LLM use | Section 14 |